# Retrieval

## Load Secrets

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Connect to the Qdrant Cluster

In [3]:
import os
import cohere
from qdrant_client import QdrantClient

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

qdrant = QdrantClient(
  url=os.environ["QDRANT_URL"],
  api_key=os.environ["QDRANT_API_KEY"]
)

COLLECTION = "week2_article"
info = qdrant.get_collection(COLLECTION)
print(f"Collection has {info.points_count} points.")
info

Collection has 91 points.


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=91, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors={'dense': VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unopt

## Utility Method For Dense Retrieval of Similar Chunks From Qdrant

In [6]:
def search(query: str, k: int = 5):
    q_vec = co.embed(
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
        texts=[query],
    ).embeddings.float[0]

    return qdrant.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        using="dense",
        limit=k
    ).points

for p in search("What is harness engineering?", k=3):
    snippet = p.payload["text"][:160].replace("\n", " ")
    print(f"[{p.id}]  {p.score:.3f}  {snippet}...")

[26]  0.473  But hold the maximalist view in your head as we walk through this section: everything you build _around_ the model is, in some sense, **harness engineering** . ...
[25]  0.452  ## **§3 - Agent Harness**   **==> picture [414 x 171] intentionally omitted <==**  Vivek Trivedy at LangChain has a one-liner I love:   " _Agent = Model + Harne...
[71]  0.434  ## **§8 - The Evaluation Harness**   **==> picture [414 x 170] intentionally omitted <==**  I'll keep this section short, because the message is short.   If you...


## Utility method to Retrieve and Generate

In [7]:
def rag_answer(question: str, k: int = 5):
    hits = search(question, k=k)
    context = "\n\n----\n\n".join(h.payload["text"] for h in hits)
    resp = co.chat(
        model="command-a-03-2025",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using ONLY the context below. If the context doesn't "
                    "cover the answer, say you don't know, Quote the article where "
                    "helpful.\n\nContext\n" + context
                ),
            },
            {
                "role": "user",
                "content": question,
            }
        ],
    )
    return resp.message.content[0].text

print(rag_answer("What is harness engineering?"))

According to the context, harness engineering refers to everything built **around** a model. As stated:

> "everything you build _around_ the model is, in some sense, **harness engineering**."

The orchestrator is considered the most central piece of harness engineering. Additionally, Vivek Trivedy's quote provides a maximalist view:

> " _Agent = Model + Harness. If you're not the model, you're the harness._ "

In the narrower definition used in the diagram, the harness specifically refers to the **orchestration loop**, while other components like tools, memory, and sandbox are treated separately in subsequent sections.


## Results When No Matching Context

In [8]:
print(rag_answer("Is Django a Python Web Development framework for building APIs?"))

The context provided does not mention Django or its role as a Python web development framework for building APIs. Therefore, I don't know the answer to this question based on the given information.


## Where This Breaks

## 4. Where this breaks

1. Pick a **rare, exact token** from your article — a specific class name, an acronym, a proper noun that appears once or twice. 
2. Dense embeddings are weak on rare tokens because they're built for meaning, not for spelling.
3. The model often misses the chunk and either invents or punts.